In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# Load toàn bộ data
sales     = pd.read_csv('sales.csv', parse_dates=['Date'])
orders    = pd.read_csv('orders.csv', parse_dates=['order_date'])
items     = pd.read_csv('order_items.csv')
products  = pd.read_csv('products.csv')
customers = pd.read_csv('customers.csv', parse_dates=['signup_date'])
geo       = pd.read_csv('geography.csv')
promos    = pd.read_csv('promotions.csv', parse_dates=['start_date','end_date'])
payments  = pd.read_csv('payments.csv')
ships     = pd.read_csv('shipments.csv', parse_dates=['ship_date','delivery_date'])
returns   = pd.read_csv('returns.csv', parse_dates=['return_date'])
reviews   = pd.read_csv('reviews.csv')
inventory = pd.read_csv('inventory.csv', parse_dates=['snapshot_date'])
traffic   = pd.read_csv('web_traffic.csv', parse_dates=['date'])

In [ ]:
# Chart 1: Revenue vs COGS chồng nhau

monthly = sales.resample('M', on='Date').sum().reset_index()
monthly['Profit_Margin'] = (monthly['Revenue'] - monthly['COGS']) / monthly['Revenue'] * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['Date'], monthly['Revenue'], label='Revenue', linewidth=2)
ax.plot(monthly['Date'], monthly['COGS'], label='COGS', linewidth=2, linestyle='--')
ax.fill_between(monthly['Date'], monthly['COGS'], monthly['Revenue'], alpha=0.15, label='Gross Profit')
ax.set_title('Revenue vs COGS theo tháng (2012–2022)', fontsize=14)
ax.set_xlabel('Tháng')
ax.set_ylabel('VND')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('chart1_revenue_cogs.png', dpi=150)
plt.show()

In [ ]:
#Chart 2: Profit Margin % theo tháng

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['Date'], monthly['Profit_Margin'], color='steelblue', linewidth=2)
ax.axhline(monthly['Profit_Margin'].mean(), color='red', linestyle='--', alpha=0.5, label=f"Trung bình: {monthly['Profit_Margin'].mean():.1f}%")
ax.set_title('Profit Margin % theo tháng (2012–2022)', fontsize=14)
ax.set_xlabel('Tháng')
ax.set_ylabel('Profit Margin (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('chart2_profit_margin.png', dpi=150)
plt.show()

# In ra tháng thấp nhất và cao nhất để viết vào phân tích
print("Tháng margin thấp nhất:")
print(monthly.nsmallest(3, 'Profit_Margin')[['Date','Profit_Margin']])
print("\nTháng margin cao nhất:")
print(monthly.nlargest(3, 'Profit_Margin')[['Date','Profit_Margin']])

In [ ]:
monthly['year']  = monthly['Date'].dt.year
monthly['month'] = monthly['Date'].dt.month

pivot = monthly.pivot(index='year', columns='month', values='Profit_Margin')

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Profit Margin (%)'})
ax.set_title('Profit Margin (%) theo Năm × Tháng', fontsize=14)
ax.set_xlabel('Tháng')
ax.set_ylabel('Năm')
plt.tight_layout()
plt.savefig('chart3_heatmap.png', dpi=150)
plt.show()

#Chart 3: Heatmap năm × tháng

In [ ]:

#Chart 4: Seasonal decompose, index là datetime
ts = monthly.set_index('Date')['Profit_Margin']
ts = ts.asfreq('M')  # đảm bảo monthly frequency

result = seasonal_decompose(ts, model='additive', period=12)

fig = result.plot()
fig.set_size_inches(14, 10)
fig.suptitle('Phân tách Profit Margin: Trend + Seasonal + Residual', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('chart4_decompose.png', dpi=150)
plt.show()

# Tìm tháng residual bất thường
residual = result.resid.dropna()
anomalies = residual[residual.abs() > residual.std() * 2]
print("Các tháng bất thường trong residual:")
print(anomalies.sort_values())

In [ ]:
#phân rã revenue/cost

# Lọc kỳ có vấn đề -.> thay bằng kỳ thật mày tìm được
problem_period = monthly[
    (monthly['year'] >= 2019) & 
    (monthly['year'] <= 2020)
].copy()

# Tính % tăng trưởng so tháng trước
problem_period['Rev_growth']  = problem_period['Revenue'].pct_change() * 100
problem_period['COGS_growth'] = problem_period['COGS'].pct_change() * 100

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(problem_period))
width = 0.35
ax.bar([i - width/2 for i in x], problem_period['Rev_growth'], width, label='Revenue growth %', color='steelblue')
ax.bar([i + width/2 for i in x], problem_period['COGS_growth'], width, label='COGS growth %', color='tomato')
ax.set_xticks(list(x))
ax.set_xticklabels(problem_period['Date'].dt.strftime('%m/%Y'), rotation=45)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Tốc độ tăng trưởng Revenue vs COGS theo tháng', fontsize=13)
ax.set_ylabel('%')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('chart5_rev_vs_cogs_growth.png', dpi=150)
plt.show()

In [ ]:
#Drill down (nếu mà có vđ ở revenue thì tính cr%)

# Gộp web traffic với orders theo ngày
daily_orders = orders.groupby('order_date').size().reset_index(name='num_orders')
traffic_orders = traffic.merge(daily_orders, left_on='date', right_on='order_date', how='left')
traffic_orders['num_orders'] = traffic_orders['num_orders'].fillna(0)
traffic_orders['CR%'] = traffic_orders['num_orders'] / traffic_orders['sessions'] * 100

# Resample theo tháng
monthly_cr = traffic_orders.resample('M', on='date')[['sessions','num_orders','CR%']].mean()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly_cr.index, monthly_cr['sessions'], alpha=0.4, label='Sessions', color='steelblue')
ax2.plot(monthly_cr.index, monthly_cr['CR%'], color='red', linewidth=2, label='CR%')
ax1.set_title('Sessions vs Conversion Rate % theo tháng', fontsize=13)
ax1.set_ylabel('Sessions')
ax2.set_ylabel('CR%')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('chart6_cr.png', dpi=150)
plt.show()

In [ ]:
#Drill down (nếu mà có vđ ở cost thì phân tích return rate)

#Join returns với products để lấy category
ret_prod = returns.merge(products[['product_id','category','size']], on='product_id')
items_prod = items.merge(products[['product_id','category','size']], on='product_id')

# Return rate theo category
ret_by_cat   = ret_prod.groupby('category').size().reset_index(name='returns')
items_by_cat = items_prod.groupby('category').size().reset_index(name='total_items')
rate_cat = ret_by_cat.merge(items_by_cat, on='category')
rate_cat['return_rate%'] = rate_cat['returns'] / rate_cat['total_items'] * 100
rate_cat = rate_cat.sort_values('return_rate%', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(rate_cat['category'], rate_cat['return_rate%'], color='tomato')
ax.axvline(rate_cat['return_rate%'].mean(), color='navy', linestyle='--',
           label=f"Trung bình: {rate_cat['return_rate%'].mean():.1f}%")
ax.set_title('Return Rate (%) theo Category', fontsize=13)
ax.set_xlabel('Return Rate (%)')
ax.legend()
plt.tight_layout()
plt.savefig('chart7_return_rate.png', dpi=150)
plt.show()

# Return reason cho category có rate cao nhất
top_cat = rate_cat.iloc[-1]['category']
reason = ret_prod[ret_prod['category'] == top_cat]['return_reason'].value_counts()
print(f"\nLý do trả hàng cho {top_cat}:")
print(reason)